## **Create Labels Using self_assessments.csv dataset**
**Create binary stress and 3-class stress lables using self_assessments dataset**

In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
# read and lod self_assessments.csv dataset
self_assessments = pd.read_csv("data/self_assessments.csv", sep = ";", header = 0, index_col = 0)
print(self_assessments.info())

print("=" *50)
self_assessments.head()

<class 'pandas.DataFrame'>
Index: 45 entries, Breathing_relax to Most_stressful
Data columns (total 65 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   hh2e    23 non-null     str  
 1   wfsl    41 non-null     str  
 2   37ir    45 non-null     str  
 3   hvpa    45 non-null     str  
 4   ql3b    45 non-null     str  
 5   r0a3    45 non-null     str  
 6   qx2o    45 non-null     str  
 7   wssm    45 non-null     str  
 8   ctzy    45 non-null     str  
 9   uyrl    45 non-null     str  
 10  v8mh    45 non-null     str  
 11  45lx    45 non-null     str  
 12  r3zm    45 non-null     str  
 13  2hpu    45 non-null     str  
 14  4woj    45 non-null     str  
 15  cxj0    45 non-null     str  
 16  tmvd    45 non-null     str  
 17  2ea4    45 non-null     str  
 18  71i5    45 non-null     str  
 19  kkf5    45 non-null     str  
 20  uymz    45 non-null     str  
 21  iqyg    45 non-null     str  
 22  9txq    45 non-null     str  
 23  kycf   

,hh2e,wfsl,37ir,hvpa,ql3b,r0a3,qx2o,wssm,ctzy,uyrl,...,y8c3,g7r2,f6q3,e5p4,d4n6,c3m7,b2l8,a1k9,t6v9,b9w0
Tasks,,,,,,,,,,,,,,,,,,,,,
Breathing_relax,6,6,9,9,10,10,9,8,9,9,...,9,8,9,10,5,4,8,7,5,8
Breathing_stress,5,4,2,1,1,0,2,0,0,1,...,1,0,1,3,2,6,1,5,5,2
Breathing_valence,NaN,8,5,7,5,8,8,6,7,6,...,5,7,7,8,5,8,7,6,6,7
Breathing_arousal,NaN,3,3,8,2,1,2,1,7,5,...,7,5,1,3,4,7,8,8,3,8
Video1_relax,8,6,3,7,10,10,9,5,4,7,...,8,8,9,8,2,2,2,7,7,5


In [3]:
self_assessments.tail()

,hh2e,wfsl,37ir,hvpa,ql3b,r0a3,qx2o,wssm,ctzy,uyrl,...,y8c3,g7r2,f6q3,e5p4,d4n6,c3m7,b2l8,a1k9,t6v9,b9w0
Tasks,,,,,,,,,,,,,,,,,,,,,
Relax_relax,6,NaN,7,6,10,10,4,7,9,8,...,9,8,9,8,5,7,6,8,3,2
Relax_stress,6,NaN,1,2,0,1,5,0,1,2,...,0,0,1,2,3,5,5,3,3,5
Relax_valence,NaN,NaN,7,6,8,6,5,7,5,5,...,6,5,8,7,5,6,7,8,5,5
Relax_arousal,NaN,NaN,6,5,8,2,3,0,1,4,...,4,2,1,3,3,7,6,5,0,4
Most_stressful,Counting2,Counting2,Speaking,Reading,Speaking,Counting3,Speaking,Speaking,Math,Speaking,...,Counting2,Counting3,Stroop,Counting2,Counting1,Stroop,Counting2,Counting2,Counting2,Counting2


In [4]:
# Store last row in a variable and drop it from primary dataset
most_stressfull = self_assessments.tail(1)
# print(most_stressfull)

df = self_assessments
df.drop(df.tail(1).index, inplace=True)

In [5]:
df.tail()

,hh2e,wfsl,37ir,hvpa,ql3b,r0a3,qx2o,wssm,ctzy,uyrl,...,y8c3,g7r2,f6q3,e5p4,d4n6,c3m7,b2l8,a1k9,t6v9,b9w0
Tasks,,,,,,,,,,,,,,,,,,,,,
Counting3_arousal,NaN,6,8,8,10,6,6,6,9,6,...,5,4,3,5,6,5,6,7,8,8
Relax_relax,6,NaN,7,6,10,10,4,7,9,8,...,9,8,9,8,5,7,6,8,3,2
Relax_stress,6,NaN,1,2,0,1,5,0,1,2,...,0,0,1,2,3,5,5,3,3,5
Relax_valence,NaN,NaN,7,6,8,6,5,7,5,5,...,6,5,8,7,5,6,7,8,5,5
Relax_arousal,NaN,NaN,6,5,8,2,3,0,1,4,...,4,2,1,3,3,7,6,5,0,4


### **Several methods to crrate stress lables from self_assessment dataset**

#### **Labels Creation Functions**

In [6]:
# -- Labeling functions --
# function to map subject/task pair values to binary class and 3-class labels for "relax", "stress", "valence", "arousal"

# 1- BINARY v1: Not stressed - Stressed
def map_to_binary(value):
    """
    0-5 -> 0 (relax) and 6-10 -> 1 (not relax)
    """
    if pd.isna(value):
        return pd.NA
    value = float(value)
    return 0 if value <= 5 else 1


# 2- BINARY v2: Not stressed - Stressed
def binary_stress_label(valence, arousal, stress, relax):
    """
    Stressed: (valence < 5 AND arousal > 5 AND stress > 5) OR (stress >= 7 AND relax < 5)
    Not stress: (valence > 5 AND arousal < 5 AND relax > 5) OR (relax >= 7 AND stress < 5)
    Returns: pd.NA when inputs are missing or the sample is ambiguous.
    """
    if any(pd.isna(v) for v in [valence, arousal, stress, relax]):
        return pd.NA
    valence, arousal, stress, relax = float(valence), float(arousal), float(stress), float(relax)
 
    if (valence < 5 and arousal > 5 and stress > 5) or (stress >= 7 and relax < 5):
        return 1  # stressed
    elif (valence > 5 and arousal < 5 and relax > 5) or (relax >= 7 and stress < 5):
        return 0  # not stressed
    return pd.NA   # ambiguous / neither condition met
 
# 3- THREE CLASS v1: Relaxed (0) - Neutral (1) - Stressed (2)
def label_3class(valence, arousal, stress, relax):
    """
    Stressed: valence < 5 AND arousal > 5 AND stress > 5
    Relaxed: valence >= 5 AND arousal <= 5 AND relax >= 5
    Neutral: everything else
    """
    if any(pd.isna(v) for v in [valence, arousal, stress, relax]):
        return pd.NA
    valence, arousal, stress, relax = float(valence), float(arousal), float(stress), float(relax)
 
    if valence < 5 and arousal > 5 and stress > 5:
        return 2  # stressed
    elif valence >= 5 and arousal <= 5 and relax >= 5:
        return 0  # relaxed
    else:
        return 1  # neutral
 
# THREE CLASS v2: (refined – strong statements get priority):
def label_3class_v2(valence, arousal, stress, relax):
    """
    Relaxed (0) - Neutral (1) - Stressed (2)
    Stressed: (valence < 5 AND arousal > 5 AND stress > 5) OR stress >= 7
    Relaxed: (valence >= 5 AND arousal <= 5 AND relax >= 5) OR relax >= 7
    Neutral: everything else
    """
    if any(pd.isna(v) for v in [valence, arousal, stress, relax]):
        return pd.NA
    valence, arousal, stress, relax = float(valence), float(arousal), float(stress), float(relax)
 
    if (valence < 5 and arousal > 5 and stress > 5) or stress >= 7:
        return 2  # stressed
    elif (valence >= 5 and arousal <= 5 and relax >= 5) or relax >= 7:
        return 0  # relaxed
    else:
        return 1  # neutral
 
# FOUR CLASS: Not stressed (0) - Neutral (1) - Mild stress (2) - Acute stress (3)
def label_4class(valence, arousal, stress, relax):
    """
    Not stressed: (valence >= 5 AND arousal <= 5 AND relax > 5) OR relax >= 7
    Acute stress: (valence <= 5 AND arousal > 5 AND stress > 5) OR stress >= 8
    Mild stress: stress > 5
    Neutral: everything else
    """
    if any(pd.isna(v) for v in [valence, arousal, stress, relax]):
        return pd.NA
    valence, arousal, stress, relax = float(valence), float(arousal), float(stress), float(relax)
 
    if (valence >= 5 and arousal <= 5 and relax > 5) or relax >= 7:
        return 0  # not stressed / relaxed
    elif (valence <= 5 and arousal > 5 and stress > 5) or stress >= 8:
        return 3  # acute stress
    elif stress > 5:
        return 2  # mild stress
    else:
        return 1  # neutral

#### **Prepare data to builing result dataset**

In [7]:
# split each row label "<Task>_<measure>" into (Task, measure)
# e.g. "Breathing_relax" -> ("Breathing", "relax")
df["__task"], df["__measure"]  = zip(*df.index.map(lambda x:x .rsplit("_", 1)))

# print(df["__task"]), print(df["__measure"])

In [8]:
df.head()

,hh2e,wfsl,37ir,hvpa,ql3b,r0a3,qx2o,wssm,ctzy,uyrl,...,f6q3,e5p4,d4n6,c3m7,b2l8,a1k9,t6v9,b9w0,__task,__measure
Tasks,,,,,,,,,,,,,,,,,,,,,
Breathing_relax,6,6,9,9,10,10,9,8,9,9,...,9,10,5,4,8,7,5,8,Breathing,relax
Breathing_stress,5,4,2,1,1,0,2,0,0,1,...,1,3,2,6,1,5,5,2,Breathing,stress
Breathing_valence,NaN,8,5,7,5,8,8,6,7,6,...,7,8,5,8,7,6,6,7,Breathing,valence
Breathing_arousal,NaN,3,3,8,2,1,2,1,7,5,...,1,3,4,7,8,8,3,8,Breathing,arousal
Video1_relax,8,6,3,7,10,10,9,5,4,7,...,9,8,2,2,2,7,7,5,Video1,relax


In [ ]:
rows = []
for subject in df.columns[:-2]:  # exclude columns __task/__measure
    for task in df["__task"].unique():
        sub_df = df[df["__task"] == task]
 
        def get(measure):
            s = sub_df.loc[sub_df["__measure"] == measure, subject]
            return s.iloc[0] if not s.empty else pd.NA
 
        relax_v = get("relax")
        stress_v = get("stress")
        valence_v = get("valence")
        arousal_v = get("arousal")
 
        rows.append({
            "subject_task": f"{subject}_{task}",

            # Original binary per-dimension columns
            "binary_relax": map_to_binary(relax_v),
            "binary_stress": map_to_binary(stress_v),
            "binary_valence": map_to_binary(valence_v),
            "binary_arousal": map_to_binary(arousal_v),

            # New multi-signal labels
            "binary_stress_label": binary_stress_label(valence_v, arousal_v, stress_v, relax_v),
            "3class_label": label_3class(valence_v, arousal_v, stress_v, relax_v),
            "3class_label_v2": label_3class_v2(valence_v, arousal_v, stress_v, relax_v),
            "4class_label": label_4class(valence_v, arousal_v, stress_v, relax_v),
        })
 
result = pd.DataFrame(rows)
result.head()

# # save the output
# result.to_csv("data/binary_labels.csv", index = False, sep = ",")

In [10]:
print("Should be :", 65*11 , ", there is:", result.shape[0])

Should be : 715 , there is: 715


In [ ]:
def class_count(result, targets):
    for target in targets:
        print(f"Class Split {target}:")
        print(result[target].value_counts())
        print("-" *50)

# list of the binary metrics
targets  = result.columns.drop("subject_task")
# targets

# call the class count function for binary labels
class_count(result, targets)

Class Split binary_relax:
binary_relax
0    385
1    329
Name: count, dtype: int64
--------------------------------------------------
Class Split binary_stress:
binary_stress
0    415
1    299
Name: count, dtype: int64
--------------------------------------------------
Class Split binary_valence:
binary_valence
0    356
1    347
Name: count, dtype: int64
--------------------------------------------------
Class Split binary_arousal:
binary_arousal
1    504
0    199
Name: count, dtype: int64
--------------------------------------------------
Class Split binary_stress_label:
binary_stress_label
0    208
1    183
Name: count, dtype: int64
--------------------------------------------------
Class Split 3class_label:
3class_label
1    473
0    138
2     92
Name: count, dtype: int64
--------------------------------------------------
Class Split 3class_label_v2:
3class_label_v2
0    259
1    243
2    201
Name: count, dtype: int64
--------------------------------------------------
Class Split 4c